<a href="https://colab.research.google.com/github/Tahleels/json-extract-qlora-dpo/blob/main/notebooks/colab_end_to_end.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QLoRA JSON Extraction: End-to-End Training Pipeline

**Stages:**
1. Install & Setup
2. Clone / Sync Project
3. Generate Datasets
4. SFT Training (QLoRA, 4-bit)
5. DPO Alignment
6. Merge Weights
7. GGUF Export

> Run cells top to bottom. Requires a T4 GPU runtime.

---
## Cell 1: Install Dependencies

In [55]:
%%bash
pip install -q \
    "transformers>=4.44" \
    "trl>=1.7" \
    "peft>=0.12" \
    "bitsandbytes>=0.43" \
    "datasets>=2.18" \
    "accelerate>=0.27" \
    pyyaml faker pydantic

import trl, transformers, peft
print(f"TRL:          {trl.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"PEFT:         {peft.__version__}")

bash: line 10: import: command not found
bash: line 11: syntax error near unexpected token `f"TRL:          {trl.__version__}"'
bash: line 11: `print(f"TRL:          {trl.__version__}")'


CalledProcessError: Command 'b'pip install -q \\\n    "transformers>=4.44" \\\n    "trl>=1.7" \\\n    "peft>=0.12" \\\n    "bitsandbytes>=0.43" \\\n    "datasets>=2.18" \\\n    "accelerate>=0.27" \\\n    pyyaml faker pydantic\n\nimport trl, transformers, peft\nprint(f"TRL:          {trl.__version__}")\nprint(f"Transformers: {transformers.__version__}")\nprint(f"PEFT:         {peft.__version__}")\n'' returned non-zero exit status 2.

---
## Cell 2: Clone / Sync Project from GitHub
Safe to run multiple times. First run clones. Subsequent runs do a force-sync to latest commit.

In [56]:
import os

REPO_DIR = "/content/json-extract-qlora-dpo"
REPO_URL = "https://github.com/Tahleels/json-extract-qlora-dpo.git"

if not os.path.exists(REPO_DIR):
    print("First run: Cloning repository...")
    %cd /content
    !git clone {REPO_URL}
else:
    print("Repository exists. Force-syncing latest from GitHub...")
    %cd {REPO_DIR}
    !git fetch origin
    !git reset --hard origin/main

%cd {REPO_DIR}
print(f"Working directory: {os.getcwd()}")

Repository exists. Force-syncing latest from GitHub...
/content/json-extract-qlora-dpo
HEAD is now at 28dedab fix: remove max_seq_length and dataset_text_field from SFTConfig (TRL 1.7 dropped both)
/content/json-extract-qlora-dpo
Working directory: /content/json-extract-qlora-dpo


---
## Cell 3: Generate Synthetic Datasets
Creates `data/sft_train.jsonl`, `data/sft_val.jsonl`, `data/sft_test.jsonl`, and `data/dpo_train.jsonl`.

In [59]:
!python -m src.data.build_sft
!python -m src.data.build_dpo

import os
for f in ["data/sft_train.jsonl", "data/sft_val.jsonl", "data/dpo_train.jsonl"]:
    size = os.path.getsize(f)
    print(f"  {f}: {size:,} bytes")
print("All datasets generated successfully!")

wrote  3400 rows -> data/sft_train.jsonl
wrote   200 rows -> data/sft_val.jsonl
wrote   400 rows -> data/sft_test.jsonl
wrote 2000 DPO pairs -> data/dpo_train.jsonl
  data/sft_train.jsonl: 1,220,978 bytes
  data/sft_val.jsonl: 71,645 bytes
  data/dpo_train.jsonl: 1,107,362 bytes
All datasets generated successfully!


---
## Cell 4: SFT Training (QLoRA, 4-bit NF4)

**What happens here:**
- Load `Qwen2.5-0.5B-Instruct` in 4-bit NF4 quantization using bitsandbytes
- Attach LoRA adapters (rank=16) to attention layers via `get_peft_model()`
- Fine-tune on 3,400 JSON extraction examples
- Save the trained adapter to `artifacts/adapter_sft`

**TRL 1.7 API notes:**
- `peft_config` removed from `SFTTrainer` → use `get_peft_model()` first
- `max_seq_length` removed from `SFTConfig` → set via `tokenizer.model_max_length`
- `dataset_text_field` not needed when the dataset is pre-formatted as `text` strings
- `tokenizer` arg → `processing_class`

In [62]:
import os, torch, yaml
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from src.config import load_config

# --- Load configs ---
cfg = load_config()
with open("configs/sft.yaml", "r") as f:
    sft_cfg = yaml.safe_load(f)

MAX_SEQ_LEN = sft_cfg["training"]["max_seq_length"]

# --- Dynamic Precision for QLoRA ---
# T4 GPUs crash with fp16=True on Qwen. L4/A100 GPUs prefer bf16=True.
# If GPU lacks native bf16, we disable Trainer precision & let bitsandbytes handle it.
use_bf16 = torch.cuda.is_bf16_supported()
print(f"GPU native bfloat16 support: {use_bf16}")

# --- 4-bit Quantization Config (NF4) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # BnB handles the actual compute precision
    bnb_4bit_use_double_quant=True,
)

# --- Load Tokenizer ---
print(f"Loading tokenizer: {cfg.base_model}")
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.model_max_length = MAX_SEQ_LEN

# --- Load Base Model in 4-bit ---
print(f"Loading model in 4-bit NF4: {cfg.base_model}")
model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# --- Wrap model with LoRA adapter BEFORE passing to SFTTrainer ---
peft_config = LoraConfig(
    r=sft_cfg["lora"]["r"],
    lora_alpha=sft_cfg["lora"]["alpha"],
    lora_dropout=sft_cfg["lora"]["dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=sft_cfg["lora"]["target_modules"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# --- Load & Format Dataset ---
dataset = load_dataset(
    "json",
    data_files={
        "train": "data/sft_train.jsonl",
        "val":   "data/sft_val.jsonl",
    }
)

def format_chat(example):
    messages = [
        {"role": "system",    "content": cfg.system_prompt},
        {"role": "user",      "content": example["input_text"]},
        {"role": "assistant", "content": example["output_json"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = dataset["train"].map(format_chat, remove_columns=dataset["train"].column_names)
val_ds   = dataset["val"].map(format_chat,   remove_columns=dataset["val"].column_names)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

# --- SFT Trainer (TRL 1.7) ---
sft_config = SFTConfig(
    output_dir=sft_cfg["training"]["output_dir"],
    num_train_epochs=sft_cfg["training"]["epochs"],
    per_device_train_batch_size=sft_cfg["training"]["batch_size"],
    gradient_accumulation_steps=sft_cfg["training"]["grad_accum"],
    learning_rate=float(sft_cfg["training"]["learning_rate"]),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="epoch",
    bf16=use_bf16,     # True if L4/A100, False if T4
    fp16=False,        # Disabled completely to avoid the GradScaler crash
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print("Starting SFT training...")
trainer.train()

os.makedirs("artifacts", exist_ok=True)
trainer.model.save_pretrained("artifacts/adapter_sft")
tokenizer.save_pretrained("artifacts/adapter_sft")
print("SFT adapter saved to: artifacts/adapter_sft")

GPU native bfloat16 support: True
Loading tokenizer: Qwen/Qwen2.5-0.5B-Instruct
Loading model in 4-bit NF4: Qwen/Qwen2.5-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
Train samples: 3400 | Val samples: 200


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting SFT training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.506831,0.506359,0.510014,233190.000000,0.900889
200,0.475122,0.478679,0.493218,466234.000000,0.904650
300,0.453341,0.468431,0.471939,698028.000000,0.905252
400,0.450756,0.462474,0.473585,931261.000000,0.905388
500,0.432807,0.459090,0.455924,1163302.000000,0.907211


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.506831,0.506359,0.510014,233190.000000,0.900889
200,0.475122,0.478679,0.493218,466234.000000,0.904650
300,0.453341,0.468431,0.471939,698028.000000,0.905252
400,0.450756,0.462474,0.473585,931261.000000,0.905388
500,0.432807,0.459090,0.455924,1163302.000000,0.907211
600,0.437650,0.456782,0.458406,1396253.000000,0.907457
639,0.437243,0.456599,0.457472,1485993.000000,0.907291


SFT adapter saved to: artifacts/adapter_sft


---
## Cell 5: DPO Alignment (Preference Tuning)

**What happens here:**
- Reload base model in 4-bit and mount the SFT adapter on top
- Run DPO with chosen (clean JSON) vs rejected (corrupted JSON) pairs
- Teaches the model to **never** output markdown fences, preambles, or trailing commas
- Save the aligned adapter to `artifacts/adapter_dpo`

> **TRL 1.7:** `tokenizer` arg renamed to `processing_class`. `ref_model=None` works when model is a PEFT model (implicit reference).

In [64]:
import os, torch, yaml
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig
from src.config import load_config

# --- Load configs ---
cfg = load_config()
with open("configs/dpo.yaml", "r") as f:
    dpo_cfg = yaml.safe_load(f)

# --- Dynamic Precision for QLoRA ---
use_bf16 = torch.cuda.is_bf16_supported()

# --- 4-bit Quantization Config ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# --- Load Tokenizer ---
print(f"Loading tokenizer: {cfg.base_model}")
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # DPO needs left-padding

# --- Load Base Model in 4-bit ---
print(f"Loading base model in 4-bit: {cfg.base_model}")
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model = prepare_model_for_kbit_training(base_model)

# --- Load SFT adapter as trainable for DPO ---
print("Loading SFT adapter for DPO...")
model = PeftModel.from_pretrained(
    base_model,
    "artifacts/adapter_sft",
    is_trainable=True,
)

# --- Load & Format DPO Dataset ---
raw_dataset = load_dataset("json", data_files={"train": "data/dpo_train.jsonl"})["train"]

def format_dpo(example):
    messages = [
        {"role": "system", "content": cfg.system_prompt},
        {"role": "user",   "content": example["prompt"]},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return {
        "prompt":   prompt_text,
        "chosen":   example["chosen"]   + tokenizer.eos_token,
        "rejected": example["rejected"] + tokenizer.eos_token,
    }

dpo_dataset = raw_dataset.map(format_dpo, remove_columns=raw_dataset.column_names)
print(f"DPO pairs: {len(dpo_dataset)}")

# --- DPO Trainer (TRL 1.7 API) ---
# FIX: Removed 'max_prompt_length' as it is deprecated in TRL 1.7+
dpo_config = DPOConfig(
    output_dir=dpo_cfg["training"]["output_dir"],
    beta=dpo_cfg["training"]["beta"],
    num_train_epochs=dpo_cfg["training"]["epochs"],
    per_device_train_batch_size=dpo_cfg["training"]["batch_size"],
    gradient_accumulation_steps=dpo_cfg["training"]["grad_accum"],
    learning_rate=float(dpo_cfg["training"]["learning_rate"]),
    max_length=dpo_cfg["training"]["max_seq_length"],
    logging_steps=10,
    save_strategy="epoch",
    bf16=use_bf16,
    fp16=False,
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

print("Starting DPO alignment...")
trainer.train()

os.makedirs("artifacts", exist_ok=True)
trainer.model.save_pretrained("artifacts/adapter_dpo")
tokenizer.save_pretrained("artifacts/adapter_dpo")
print("DPO adapter saved to: artifacts/adapter_dpo")

Loading tokenizer: Qwen/Qwen2.5-0.5B-Instruct
Loading base model in 4-bit: Qwen/Qwen2.5-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading SFT adapter for DPO...
DPO pairs: 2000


Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting DPO alignment...


Step,Training Loss
10,0.575745
20,0.383678
30,0.314570
40,0.261472
50,0.222967
60,0.147622
70,0.141215
80,0.121904
90,0.118466
100,0.112850


DPO adapter saved to: artifacts/adapter_dpo


---
## Cell 6: Merge LoRA Weights

**What happens here:**
- Load the base model in full FP16 (no quantization — needed for a clean merge)
- Apply the DPO adapter on top
- Call `merge_and_unload()` to fold adapter weights permanently into the base model
- Save the resulting standalone model to `artifacts/merged_model`

In [66]:
# Colab Fix: Uninstall incompatible torchao version (not needed for FP16 merging)
!pip uninstall -y torchao

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.config import load_config

cfg = load_config()

print(f"Loading base model in FP16 (no quantization): {cfg.base_model}")
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("Applying DPO adapter...")
model = PeftModel.from_pretrained(base_model, "artifacts/adapter_dpo")

print("Merging adapter weights into base model...")
merged_model = model.merge_and_unload()

print("Saving merged model...")
merged_model.save_pretrained("artifacts/merged_model")
tokenizer.save_pretrained("artifacts/merged_model")
print("Done! Merged model saved to: artifacts/merged_model")

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Loading base model in FP16 (no quantization): Qwen/Qwen2.5-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Applying DPO adapter...
Merging adapter weights into base model...
Saving merged model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done! Merged model saved to: artifacts/merged_model


---
## Cell 7: GGUF Export & Quantization

**What happens here:**
- Clone and compile `llama.cpp` using cmake
- Convert the merged HF model to GGUF FP16 format
- Quantize to `Q4_K_M` (~400MB) — ready to run locally on a CPU laptop

In [67]:
import os

# Clone and build llama.cpp
if not os.path.exists("llama.cpp"):
    !git clone https://github.com/ggerganov/llama.cpp.git
!cmake llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF
!cmake --build llama.cpp/build --config Release -j$(nproc)
!pip install -q -r llama.cpp/requirements.txt

# Convert HF -> GGUF FP16
!python llama.cpp/convert_hf_to_gguf.py \
    artifacts/merged_model \
    --outfile artifacts/model-f16.gguf \
    --outtype f16

# Quantize to Q4_K_M
!./llama.cpp/build/bin/llama-quantize \
    artifacts/model-f16.gguf \
    artifacts/model-Q4_K_M.gguf \
    Q4_K_M

size_mb = os.path.getsize("artifacts/model-Q4_K_M.gguf") / 1024 / 1024
print(f"GGUF model ready: artifacts/model-Q4_K_M.gguf ({size_mb:.1f} MB)")

Cloning into 'llama.cpp'...
remote: Enumerating objects: 101583, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 101583 (delta 0), reused 0 (delta 0), pack-reused 101577 (from 1)
Receiving objects: 100% (101583/101583), 403.49 MiB | 18.49 MiB/s, done.
Resolving deltas: 100% (71515/71515), done.
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assemble

FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/model-Q4_K_M.gguf'

In [68]:
import os
import json

# --- FIX: Patch tokenizer config for llama.cpp compatibility ---
# This temporarily fixes the JSON file saved by Cell 6 so llama.cpp can read it
config_path = "artifacts/merged_model/tokenizer_config.json"
with open(config_path, "r") as f:
    tok_config = json.load(f)

if "extra_special_tokens" in tok_config:
    del tok_config["extra_special_tokens"]
    with open(config_path, "w") as f:
        json.dump(tok_config, f, indent=2)
    print("Patched tokenizer_config.json successfully.")
else:
    print("Tokenizer config is already clean.")

# Clone and build llama.cpp (will skip cloning since it already exists)
if not os.path.exists("llama.cpp"):
    !git clone https://github.com/ggerganov/llama.cpp.git
!cmake llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF
!cmake --build llama.cpp/build --config Release -j$(nproc)
!pip install -q -r llama.cpp/requirements.txt

# Convert HF -> GGUF FP16
!python llama.cpp/convert_hf_to_gguf.py \
    artifacts/merged_model \
    --outfile artifacts/model-f16.gguf \
    --outtype f16

# Quantize to Q4_K_M
!./llama.cpp/build/bin/llama-quantize \
    artifacts/model-f16.gguf \
    artifacts/model-Q4_K_M.gguf \
    Q4_K_M

size_mb = os.path.getsize("artifacts/model-Q4_K_M.gguf") / 1024 / 1024
print(f"\nGGUF model ready: artifacts/model-Q4_K_M.gguf ({size_mb:.1f} MB)")

# Automatically download to your local computer
from google.colab import files
print("Starting download to your local machine...")
files.download("artifacts/model-Q4_K_M.gguf")

Patched tokenizer_config.json successfully.
CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.15.3
-- ggml commit:  fdb1db877
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (0.8s)
-- Generating done (0.7s)
-- Build files have been written to: /content/json-extract-qlora-dpo/llama.cpp/build
[  1%] Built target llama-common-base
[  3%] Built target ggml-base
[  3%] Built target cpp-httplib
[  3%] Built target sha256
[  4%] Built target xxhash
[  4%] Built target sha1
[  4%] Built target llama-llava-cli
[  4%] Built target llama-ui-embed
[  4%] Built target llama-minicpmv-cli
[  4%] Built target llama-gemma3-cli
[  4%] Built target llama-qwen2vl-cli
[  4%] Provisioning

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>